# 06 · Temporal-Difference Learning
**TABULAR METHODS** · ⏱ 50 min · 🧪 4 exercises · ⭐ 570 XP

*Learn a guess from a guess — TD(0), SARSA, Q-learning, Expected SARSA and Double Q-learning, with the cliff as the proving ground.*

> **First time?** Open **00 · START HERE** in the same folder. Then run the cell below to connect this notebook to your Drive folder — your progress and XP are saved there.

In [ ]:
#@title ▶ **Run me first** — connect this notebook to your Reward Signal folder
#@markdown Mounts Google Drive and finds the `RewardSignal` folder (progress, XP and your files live there).
#@markdown Leave `ROOT` empty unless you put the folder somewhere unusual, e.g. `/content/drive/MyDrive/courses/RewardSignal`.
ROOT = ""  #@param {type:"string"}
LESSON = "06-td-learning"
import os, sys

def _rs_locate(root=""):
    tries = [root, os.environ.get("RS_ROOT", "")]
    try:
        from google.colab import drive  # only exists in Colab
        if not os.path.isdir("/content/drive/MyDrive"):
            drive.mount("/content/drive")
        base = "/content/drive/MyDrive"
        tries += [f"{base}/RewardSignal", f"{base}/Colab Notebooks/RewardSignal", f"{base}/RewardSignal/RewardSignal"]
        tries += [os.path.join(base, d, "RewardSignal") for d in sorted(os.listdir(base)) if not d.startswith(".")]
        tries += ["/content/RewardSignal"]
    except ImportError:  # local Jupyter: the folder this notebook sits in
        tries += [os.getcwd(), os.path.dirname(os.getcwd())]
    for t in tries:
        if t and os.path.isfile(os.path.join(t, "runtime", "rlcolab.py")):
            return os.path.abspath(t)
    raise FileNotFoundError("Couldn't find the RewardSignal folder. Upload it to My Drive "
                            "(see its README.md) or type its full path into ROOT, then run this cell again.")

RS_ROOT = _rs_locate(ROOT.strip())
sys.dont_write_bytecode = True  # no __pycache__ folders in your Drive
if os.path.join(RS_ROOT, "runtime") not in sys.path:
    sys.path.insert(0, os.path.join(RS_ROOT, "runtime"))
import rlcolab
rlcolab.setup(RS_ROOT, lesson=LESSON)

> If one had to identify one idea as central and novel to reinforcement learning, it would
> undoubtedly be temporal-difference learning. — Sutton & Barto

Monte Carlo waits until the end of the episode to find out what a state was worth. Dynamic
programming never waits, but needs the model. **Temporal-difference (TD) learning** takes the
best of both: like MC it learns from raw experience, and like DP it **bootstraps** — it updates
an estimate using other estimates, one step at a time.

## 01 · The driving-home example

You leave the office and predict *30 minutes* to get home. Five minutes in, it starts raining;
traffic will be slow, and you now predict *35 more minutes* — a total of 40.

- **Monte Carlo** says: wait until you are home, see that it took 43, *then* correct the guess
  you made at the office.
- **TD** says: you already have new information. Your office estimate (30) and your current
  estimate (5 + 35 = 40) disagree by 10 minutes. Move the office estimate towards 40 **now**.

Every step gives you a slightly better-informed guess about the same quantity. TD learns from
the *difference between successive predictions* — hence the name.

## 02 · TD(0) prediction

Recall the two forms of v_π from the DP lesson:

```text
v_π(s) = E_π[ Gₜ | Sₜ = s ] = E_π[ Rₜ₊₁ + γ v_π(Sₜ₊₁) | Sₜ = s ]
```

MC targets the first form: the sampled return Gₜ. TD targets the second, replacing the unknown
v_π(Sₜ₊₁) with the current estimate V(Sₜ₊₁):

```text
V(Sₜ) ← V(Sₜ) + α [ Rₜ₊₁ + γ V(Sₜ₊₁) − V(Sₜ) ]
```

The bracket is the **TD error**, the single most important quantity in this course:

```text
δₜ = Rₜ₊₁ + γ V(Sₜ₊₁) − V(Sₜ)
```

δ > 0 means "things turned out better than I expected"; δ < 0, worse. If Sₜ₊₁ is terminal its
value is 0 by definition, so δ = Rₜ₊₁ − V(Sₜ).

| | target | needs a model? | bootstraps? | when can it update? |
|---|---|---|---|---|
| DP | E[R + γ V(S')] over all s' | yes | yes | any time (sweeps) |
| MC | Gₜ (full sampled return) | no | no | end of episode |
| TD(0) | Rₜ₊₁ + γ V(Sₜ₊₁) (one sample) | no | yes | after every step |

> **🔑 KEY IDEA** · **The TD error**
>
> δ = R + γ V(S') − V(S) compares what you predicted before a step with a better-informed
> prediction made one step later. Nudging V by α·δ is how TD learns — and δ reappears in SARSA,
> Q-learning, actor-critic, GAE and PPO.

### A hand-worked chain

Three states 0 → 1 → 2 → terminal; the only reward is +1 on the final step; γ = 1, α = 0.5,
V starts at 0.

- **Episode 1.** Steps from 0 and 1 have δ = 0 + 0 − 0 = 0: nothing changes. From 2:
  δ = 1 − 0 = 1, so V(2) = 0.5.
- **Episode 2.** From 1: δ = 0 + V(2) − V(1) = 0.5, so V(1) = 0.25. From 2: δ = 1 − 0.5, V(2) = 0.75.

The reward percolates backwards **one state per episode**. Monte Carlo would have set all three
states to 0.5 after the very first episode. Is TD slower, then? Not in general — see below.

In [ ]:
#@title 🧠 **Knowledge check 1** · +25 XP — run, then pick an answer
rlcolab.quiz("06-td-learning#0", {"q": "V(S) = 2, V(S') = 5, R = −1, γ = 0.9, α = 0.1. What is V(S) after one TD(0) update?", "options": ["2.15", "2.35", "2.6", "1.85"], "answer": 0, "explain": "δ = −1 + 0.9·5 − 2 = 1.5, so V(S) ← 2 + 0.1·1.5 = 2.15."})

### The random walk

Sutton & Barto's Example 6.2: five states A–E in a row, start in C, step left or right with
equal probability. Falling off the right end pays +1, the left end 0. The true values are
1/6, 2/6, …, 5/6 — the probability of exiting right.

---
### 🧪 Exercise 1 · TD(0) on the random walk · ⭐ **120 XP**

Implement temporal-difference prediction — learning a guess from a guess.

1. `td_error(r, v_s, v_next, gamma, terminated)` returns **δ = r + γ·V(S') − V(S)**, where a terminal S' is worth 0 (so δ = r − V(S) when `terminated`).
2. `td0(env, policy, n_episodes, alpha, gamma=1.0, init=0.0, V=None)` returns a numpy array `V` of length `env.n_states`. If `V` is `None` create `np.full(env.n_states, init)`, otherwise keep updating the given array **in place**. After **every** step apply `V[s] += alpha * δ`. Stop an episode on `terminated or truncated`.

`RandomWalk` (Sutton & Barto Example 6.2), the `coin_flip` policy and a `constant_alpha_mc` for comparison are provided. The check starts with a 3-step deterministic chain where you can compute every number by hand, then measures your RMS error on the random walk. **Run** reproduces Figure 6.2: TD vs MC learning curves.

> ✏️ **Edit** the next cell and run it (`Shift+Enter`) — that also runs its demo. Then run **✅ Check**. Stuck? The hints are right below the check.

In [ ]:
# ✏️ EXERCISE 06-td-learning/01-td0-random-walk
import random

import numpy as np
import rlhud


def td_error(r, v_s, v_next, gamma, terminated):
    """delta = R + gamma * V(S') - V(S).  A terminal S' has value 0 by definition."""
    # TODO
    raise NotImplementedError


def td0(env, policy, n_episodes, alpha, gamma=1.0, init=0.0, V=None):
    """Tabular TD(0) prediction of v_pi.

    Returns V, a numpy array of length env.n_states. If V is None, start every entry at `init`;
    otherwise keep updating the given array in place (handy to continue training).
    After EVERY step:  V(S) <- V(S) + alpha * delta
    """
    if V is None:
        V = np.full(env.n_states, float(init))
    for _ in range(n_episodes):
        # TODO: reset, then after EVERY step update V[s] with the TD error.
        raise NotImplementedError
    return V


# ---------------------------------------------------------------- provided
class RandomWalk:
    """Sutton & Barto Example 6.2.  Terminal 0  |  A B C D E  |  terminal 6.

    States 1..n are non-terminal; start in the middle. Action 0 = left, 1 = right.
    Stepping off the right end gives `right_reward` (default +1), off the left end `left_reward`.
    """

    def __init__(self, n=5, left_reward=0.0, right_reward=1.0):
        self.n = n
        self.n_states = n + 2
        self.n_actions = 2
        self.left_reward, self.right_reward = left_reward, right_reward

    def reset(self, seed=None):
        self.s = (self.n + 1) // 2
        return self.s, {}

    def step(self, a):
        self.s += 1 if a == 1 else -1
        if self.s == 0:
            return 0, self.left_reward, True, False, {}
        if self.s == self.n + 1:
            return self.s, self.right_reward, True, False, {}
        return self.s, 0.0, False, False, {}

    def true_values(self):
        """Exact v_pi for the 50/50 policy (non-terminal states 1..n)."""
        k = np.arange(1, self.n + 1)
        return self.left_reward + (self.right_reward - self.left_reward) * k / (self.n + 1)


def coin_flip(s):
    return random.randrange(2)


def constant_alpha_mc(env, policy, n_episodes, alpha, gamma=1.0, init=0.0, V=None):
    """For comparison: V(S) <- V(S) + alpha * (G - V(S)), applied at the END of each episode."""
    if V is None:
        V = np.full(env.n_states, float(init))
    for _ in range(n_episodes):
        s, _ = env.reset()
        traj = []
        while True:
            s2, r, term, trunc, _ = env.step(policy(s))
            traj.append((s, r))
            if term or trunc:
                break
            s = s2
        G = 0.0
        for s, r in reversed(traj):
            G = r + gamma * G
            V[s] += alpha * (G - V[s])
    return V


def rms(V, env):
    return float(np.sqrt(np.mean((V[1:-1] - env.true_values()) ** 2)))


if __name__ == "__main__":
    random.seed(0)
    env = RandomWalk()

    # 1. Estimates after 1, 10, 100 episodes (S&B Figure 6.2, left)
    for n in (1, 10, 100):
        random.seed(n)
        V = td0(env, coin_flip, n, alpha=0.1, init=0.5)
        rlhud.plot(V[1:-1], label=f"{n} episodes", title="TD(0) estimates of states A–E, α = 0.1")
    rlhud.plot(env.true_values(), label="true", title="TD(0) estimates of states A–E, α = 0.1")

    # 2. RMS error per episode, averaged over runs: TD vs MC (Figure 6.2, right)
    RUNS, EPISODES = 50, 100
    for name, fn, alpha in [("TD α=0.05", td0, 0.05), ("TD α=0.15", td0, 0.15),
                            ("MC α=0.01", constant_alpha_mc, 0.01), ("MC α=0.04", constant_alpha_mc, 0.04)]:
        err = np.zeros(EPISODES)
        for run in range(RUNS):
            random.seed(1000 + run)
            V = np.full(env.n_states, 0.5)
            for ep in range(EPISODES):
                fn(env, coin_flip, 1, alpha, V=V)      # one more episode, in place
                err[ep] += rms(V, env) / RUNS
        rlhud.plot(err, label=name, title="RMS error vs episodes (avg of 50 runs)",
                   xlabel="episodes", ylabel="RMS error")
        rlhud.stat(f"Final RMS, {name}", round(float(err[-1]), 3))
    print("TD learns faster than MC on this task at every step size shown.")

In [ ]:
#@title ✅ **Check** · TD(0) on the random walk (+120 XP) — tests the code you last ran above
rlcolab.check("06-td-learning/01-td0-random-walk")

<details><summary>💡 <b>Hint 1</b> of 3 — try it before opening the next one</summary>

The episode loop is the same as in every lesson so far: `s, _ = env.reset()`, then `s2, r, terminated, truncated, _ = env.step(policy(s))`. The only new thing is the update you do right after each step.


<details><summary>💡 <b>Hint 2</b> of 3 — try it before opening the next one</summary>

Use your own `td_error`: `V[s] += alpha * td_error(r, V[s], V[s2], gamma, terminated)`. Then `s = s2` and continue unless the episode is over.


<details><summary>💡 <b>Hint 3</b> of 3 — nearly gives it away</summary>

def td_error(r, v_s, v_next, gamma, terminated):
    target = r if terminated else r + gamma * v_next
    return target - v_s

and in td0: `if V is None: V = np.full(env.n_states, float(init))` before the episode loop; `return V` at the end.

</details>

</details>

</details>

In [ ]:
#@title 🔒 Reference solution — double-click to reveal · running it replaces your version
rlcolab.solution_gate("06-td-learning/01-td0-random-walk")  # obeys the Solutions setting in 00 · START HERE
# ✏️ EXERCISE 06-td-learning/01-td0-random-walk
# Reference solution
import random

import numpy as np
import rlhud


def td_error(r, v_s, v_next, gamma, terminated):
    """delta = R + gamma * V(S') - V(S).  A terminal S' has value 0 by definition."""
    target = r if terminated else r + gamma * v_next
    return target - v_s


def td0(env, policy, n_episodes, alpha, gamma=1.0, init=0.0, V=None):
    """Tabular TD(0) prediction of v_pi.

    Returns V, a numpy array of length env.n_states. If V is None, start every entry at `init`;
    otherwise keep updating the given array in place (handy to continue training).
    After EVERY step:  V(S) <- V(S) + alpha * delta
    """
    if V is None:
        V = np.full(env.n_states, float(init))
    for _ in range(n_episodes):
        s, _ = env.reset()
        while True:
            s2, r, terminated, truncated, _ = env.step(policy(s))
            V[s] += alpha * td_error(r, V[s], V[s2], gamma, terminated)
            if terminated or truncated:
                break
            s = s2
    return V


# ---------------------------------------------------------------- provided
class RandomWalk:
    """Sutton & Barto Example 6.2.  Terminal 0  |  A B C D E  |  terminal 6.

    States 1..n are non-terminal; start in the middle. Action 0 = left, 1 = right.
    Stepping off the right end gives `right_reward` (default +1), off the left end `left_reward`.
    """

    def __init__(self, n=5, left_reward=0.0, right_reward=1.0):
        self.n = n
        self.n_states = n + 2
        self.n_actions = 2
        self.left_reward, self.right_reward = left_reward, right_reward

    def reset(self, seed=None):
        self.s = (self.n + 1) // 2
        return self.s, {}

    def step(self, a):
        self.s += 1 if a == 1 else -1
        if self.s == 0:
            return 0, self.left_reward, True, False, {}
        if self.s == self.n + 1:
            return self.s, self.right_reward, True, False, {}
        return self.s, 0.0, False, False, {}

    def true_values(self):
        """Exact v_pi for the 50/50 policy (non-terminal states 1..n)."""
        k = np.arange(1, self.n + 1)
        return self.left_reward + (self.right_reward - self.left_reward) * k / (self.n + 1)


def coin_flip(s):
    return random.randrange(2)


def constant_alpha_mc(env, policy, n_episodes, alpha, gamma=1.0, init=0.0, V=None):
    """For comparison: V(S) <- V(S) + alpha * (G - V(S)), applied at the END of each episode."""
    if V is None:
        V = np.full(env.n_states, float(init))
    for _ in range(n_episodes):
        s, _ = env.reset()
        traj = []
        while True:
            s2, r, term, trunc, _ = env.step(policy(s))
            traj.append((s, r))
            if term or trunc:
                break
            s = s2
        G = 0.0
        for s, r in reversed(traj):
            G = r + gamma * G
            V[s] += alpha * (G - V[s])
    return V


def rms(V, env):
    return float(np.sqrt(np.mean((V[1:-1] - env.true_values()) ** 2)))


if __name__ == "__main__":
    random.seed(0)
    env = RandomWalk()

    # 1. Estimates after 1, 10, 100 episodes (S&B Figure 6.2, left)
    for n in (1, 10, 100):
        random.seed(n)
        V = td0(env, coin_flip, n, alpha=0.1, init=0.5)
        rlhud.plot(V[1:-1], label=f"{n} episodes", title="TD(0) estimates of states A–E, α = 0.1")
    rlhud.plot(env.true_values(), label="true", title="TD(0) estimates of states A–E, α = 0.1")

    # 2. RMS error per episode, averaged over runs: TD vs MC (Figure 6.2, right)
    RUNS, EPISODES = 50, 100
    for name, fn, alpha in [("TD α=0.05", td0, 0.05), ("TD α=0.15", td0, 0.15),
                            ("MC α=0.01", constant_alpha_mc, 0.01), ("MC α=0.04", constant_alpha_mc, 0.04)]:
        err = np.zeros(EPISODES)
        for run in range(RUNS):
            random.seed(1000 + run)
            V = np.full(env.n_states, 0.5)
            for ep in range(EPISODES):
                fn(env, coin_flip, 1, alpha, V=V)      # one more episode, in place
                err[ep] += rms(V, env) / RUNS
        rlhud.plot(err, label=name, title="RMS error vs episodes (avg of 50 runs)",
                   xlabel="episodes", ylabel="RMS error")
        rlhud.stat(f"Final RMS, {name}", round(float(err[-1]), 3))
    print("TD learns faster than MC on this task at every step size shown.")

On this task TD reaches a lower error than constant-α MC for every step size tried, and it
gets there faster. Why? TD exploits the **Markov property**: a state's value only depends on
what happens next, and TD's target uses everything already learned about the next state.
MC treats every return as an independent fact and ignores that structure.

> **ℹ️ NOTE** · **Bias and variance**
>
> The MC target Gₜ is unbiased but noisy — it depends on *every* random event until the episode
> ends. The TD target R + γV(S') is biased (V(S') is just a guess) but depends on only one random
> transition, so its variance is much lower. Lower variance usually wins in practice. Lesson 7
> turns this into a dial you can set anywhere between the two.

## 03 · SARSA: on-policy TD control

For control we switch to action values — the reason is the same as in the MC lesson: acting
greedily on Q needs no model. The TD update for Q, applied to the quintuple
(Sₜ, Aₜ, Rₜ₊₁, Sₜ₊₁, Aₜ₊₁), gives the algorithm its name:

```text
Q(S, A) ← Q(S, A) + α [ R + γ Q(S', A') − Q(S, A) ]
```

A' is the action the agent **will actually take** next, drawn from the same ε-greedy policy
it is following. So SARSA evaluates its own behaviour, exploration included, and improves it at
the same time: GPI with one-step evaluations.

```
choose A from S (ε-greedy on Q)
loop:
    take A, observe R, S'
    choose A' from S' (ε-greedy on Q)
    Q(S,A) += α [R + γ Q(S',A') − Q(S,A)]
    S, A ← S', A'
```

### Cliff walking

`rlenvs.CliffWalking` (Example 6.6) is a 4 × 12 grid. Start bottom-left, goal bottom-right,
and the cells between are a cliff: stepping in costs −100 and sends you back to the start.
Every other step is −1. The shortest path hugs the cliff edge (13 steps, return −13).

Press play and watch SARSA learn. The `C` cells are the cliff.

In [ ]:
#@title 🎮 **Interactive** · Live TD control
rlcolab.widget("qlearning", map=["............", "............", "............", "SCCCCCCCCCCG"], algo="sarsa", alpha=0.5, gamma=1.0, epsilon=0.1)

---
### 🧪 Exercise 2 · SARSA on the cliff · ⭐ **150 XP**

Build on-policy TD control and watch it play it safe.

1. `epsilon_greedy(Q, s, epsilon, rng)` — `Q` is a `(n_states, n_actions)` array. With probability ε return a random action, otherwise a greedy one, **breaking ties at random** (`rng.choice(np.flatnonzero(Q[s] == Q[s].max()))`). With an all-zero table, `np.argmax` alone would always pick UP.
2. `sarsa_update(Q, s, a, r, s2, a2, alpha, gamma, terminated)` — in place, `Q[s,a] += α·(r + γ·Q[s2,a2] − Q[s,a])`; if `terminated` the target is just `r`. Return δ.
3. `sarsa(env, n_episodes, alpha=0.5, gamma=1.0, epsilon=0.1, seed=0)` → `(Q, returns)`. Use `rng = np.random.default_rng(seed)` and `Q = np.zeros((env.n_states, env.n_actions))`. Per episode: choose A from S; then loop: step, choose A' from S', update, move on with `s, a = s2, a2`. Stop on `terminated or truncated`. `returns[i]` is episode *i*'s total reward.

`greedy_rollout(env, Q)` is provided. The check trains on `CliffWalking` (5 seeds, α = 0.1) and expects the **safe** path (return −15 to −19), not the cliff-edge one.

> ✏️ **Edit** the next cell and run it (`Shift+Enter`) — that also runs its demo. Then run **✅ Check**. Stuck? The hints are right below the check.

In [ ]:
# ✏️ EXERCISE 06-td-learning/02-sarsa-cliff
import numpy as np
import rlhud
from rlenvs import CliffWalking


def epsilon_greedy(Q, s, epsilon, rng):
    """Action for state s from the table Q (shape n_states x n_actions).
    With prob. epsilon a uniformly random action, else a greedy one (ties broken at random)."""
    # TODO
    raise NotImplementedError


def sarsa_update(Q, s, a, r, s2, a2, alpha, gamma, terminated):
    """One SARSA update, in place:  Q(S,A) += alpha * [R + gamma*Q(S',A') - Q(S,A)].
    If terminated, the target is just R. Returns the TD error delta."""
    # TODO
    raise NotImplementedError


def sarsa(env, n_episodes, alpha=0.5, gamma=1.0, epsilon=0.1, seed=0):
    """On-policy TD control. Returns (Q, returns) where returns[i] is the total reward of episode i."""
    rng = np.random.default_rng(seed)
    Q = np.zeros((env.n_states, env.n_actions))
    returns = []
    for _ in range(n_episodes):
        # TODO: choose A from S, then loop: step, choose A' from S', sarsa_update, (s, a) <- (s', a')
        #       stop on terminated or truncated; append the episode's total reward to `returns`
        raise NotImplementedError
    return Q, returns


# ---------------------------------------------------------------- provided
def greedy_rollout(env, Q, max_steps=100):
    """Follow argmax Q from the start (no exploration). Returns (total_reward, visited_states)."""
    s, _ = env.reset()
    total, path = 0.0, [s]
    for _ in range(max_steps):
        s, r, term, trunc, _ = env.step(int(np.argmax(Q[s])))
        total += r
        path.append(s)
        if term:
            break
    return total, path


def arrows(Q):
    """Greedy action per cell for rlhud.grid, with no arrow on the cliff and the goal."""
    g = np.argmax(Q, axis=1).reshape(4, 12).tolist()
    for c in range(1, 12):
        g[3][c] = None
    return g


if __name__ == "__main__":
    env = CliffWalking()
    live = rlhud.chart("SARSA on the cliff: return per episode", smooth=20, ylabel="return")
    Q, rets = sarsa(env, 500, alpha=0.5, epsilon=0.1, seed=0)
    for G in rets:
        live.add("SARSA (ε = 0.1)", max(G, -150))
    live.flush()

    G, path = greedy_rollout(env, Q)
    visits = np.zeros(env.n_states)
    visits[path] = 1
    rlhud.grid(env.to_grid(Q.max(axis=1)), title="max_a Q(s, a) and greedy policy",
               policy=arrows(Q), labels=env.labels())
    rlhud.grid(env.to_grid(visits), title=f"Greedy path ({len(path) - 1} steps, return {G:.0f})",
               labels=env.labels(), cmap="heat", fmt=".0f")
    rlhud.stat("Greedy path return", G)
    rlhud.stat("Mean online return (last 100)", round(float(np.mean(rets[-100:])), 1))
    print("SARSA learns the SAFE path: it knows its own ε-greedy steps sometimes slip into the cliff.")

In [ ]:
#@title ✅ **Check** · SARSA on the cliff (+150 XP) — tests the code you last ran above
rlcolab.check("06-td-learning/02-sarsa-cliff")

<details><summary>💡 <b>Hint 1</b> of 3 — try it before opening the next one</summary>

The name is the update: S, A, R, S', A'. You must pick A' *before* updating, use it in the target, and then actually take it on the next step — do not pick a fresh action at the top of the loop.


<details><summary>💡 <b>Hint 2</b> of 3 — try it before opening the next one</summary>

Skeleton:

s, _ = env.reset(); a = epsilon_greedy(Q, s, epsilon, rng); G = 0.0
while True:
    s2, r, terminated, truncated, _ = env.step(a); G += r
    a2 = epsilon_greedy(Q, s2, epsilon, rng)
    ...


<details><summary>💡 <b>Hint 3</b> of 3 — nearly gives it away</summary>

...then `sarsa_update(Q, s, a, r, s2, a2, alpha, gamma, terminated)`; `if terminated or truncated: break`; `s, a = s2, a2`. After the loop, `returns.append(G)`. sarsa_update itself: `target = r if terminated else r + gamma * Q[s2, a2]; delta = target - Q[s, a]; Q[s, a] += alpha * delta; return delta`.

</details>

</details>

</details>

In [ ]:
#@title 🔒 Reference solution — double-click to reveal · running it replaces your version
rlcolab.solution_gate("06-td-learning/02-sarsa-cliff")  # obeys the Solutions setting in 00 · START HERE
# ✏️ EXERCISE 06-td-learning/02-sarsa-cliff
# Reference solution
import numpy as np
import rlhud
from rlenvs import CliffWalking


def epsilon_greedy(Q, s, epsilon, rng):
    """Action for state s from the table Q (shape n_states x n_actions).
    With prob. epsilon a uniformly random action, else a greedy one (ties broken at random)."""
    if rng.random() < epsilon:
        return int(rng.integers(Q.shape[1]))
    best = np.flatnonzero(Q[s] == Q[s].max())
    return int(rng.choice(best))


def sarsa_update(Q, s, a, r, s2, a2, alpha, gamma, terminated):
    """One SARSA update, in place:  Q(S,A) += alpha * [R + gamma*Q(S',A') - Q(S,A)].
    If terminated, the target is just R. Returns the TD error delta."""
    target = r if terminated else r + gamma * Q[s2, a2]
    delta = target - Q[s, a]
    Q[s, a] += alpha * delta
    return delta


def sarsa(env, n_episodes, alpha=0.5, gamma=1.0, epsilon=0.1, seed=0):
    """On-policy TD control. Returns (Q, returns) where returns[i] is the total reward of episode i."""
    rng = np.random.default_rng(seed)
    Q = np.zeros((env.n_states, env.n_actions))
    returns = []
    for _ in range(n_episodes):
        s, _ = env.reset()
        a = epsilon_greedy(Q, s, epsilon, rng)
        G = 0.0
        while True:
            s2, r, terminated, truncated, _ = env.step(a)
            G += r
            a2 = epsilon_greedy(Q, s2, epsilon, rng)    # choose A' FIRST: it is used in the target...
            sarsa_update(Q, s, a, r, s2, a2, alpha, gamma, terminated)
            if terminated or truncated:
                break
            s, a = s2, a2                                # ...and it is the action we actually take next
        returns.append(G)
    return Q, returns


# ---------------------------------------------------------------- provided
def greedy_rollout(env, Q, max_steps=100):
    """Follow argmax Q from the start (no exploration). Returns (total_reward, visited_states)."""
    s, _ = env.reset()
    total, path = 0.0, [s]
    for _ in range(max_steps):
        s, r, term, trunc, _ = env.step(int(np.argmax(Q[s])))
        total += r
        path.append(s)
        if term:
            break
    return total, path


def arrows(Q):
    """Greedy action per cell for rlhud.grid, with no arrow on the cliff and the goal."""
    g = np.argmax(Q, axis=1).reshape(4, 12).tolist()
    for c in range(1, 12):
        g[3][c] = None
    return g


if __name__ == "__main__":
    env = CliffWalking()
    live = rlhud.chart("SARSA on the cliff: return per episode", smooth=20, ylabel="return")
    Q, rets = sarsa(env, 500, alpha=0.5, epsilon=0.1, seed=0)
    for G in rets:
        live.add("SARSA (ε = 0.1)", max(G, -150))
    live.flush()

    G, path = greedy_rollout(env, Q)
    visits = np.zeros(env.n_states)
    visits[path] = 1
    rlhud.grid(env.to_grid(Q.max(axis=1)), title="max_a Q(s, a) and greedy policy",
               policy=arrows(Q), labels=env.labels())
    rlhud.grid(env.to_grid(visits), title=f"Greedy path ({len(path) - 1} steps, return {G:.0f})",
               labels=env.labels(), cmap="heat", fmt=".0f")
    rlhud.stat("Greedy path return", G)
    rlhud.stat("Mean online return (last 100)", round(float(np.mean(rets[-100:])), 1))
    print("SARSA learns the SAFE path: it knows its own ε-greedy steps sometimes slip into the cliff.")

SARSA does not take the shortest path. It walks up and around, two or three rows away from the
edge. That is not a bug: SARSA learns the value of the policy it *follows*, and that policy is
ε-greedy. Next to the cliff, one random step in ten is a disaster, so cells next to the edge
really are worth less **for an ε-greedy agent**.

## 04 · Q-learning: off-policy TD control

Change one term — bootstrap from the **best** next action instead of the one you will take:

```text
Q(S, A) ← Q(S, A) + α [ R + γ maxₐ' Q(S', a') − Q(S, A) ]
```

Now the learned Q approximates q*, the optimal action values, **no matter how the agent
behaves** (as long as every pair keeps being visited). The behaviour policy is ε-greedy; the
target policy is greedy. That is off-policy learning — without importance sampling. Why are no
ratios needed? Because the target for Q(S, A) only involves the next action, and the max takes
the place of the expectation over the target policy; the action A itself is already given.

In [ ]:
#@title 🎮 **Interactive** · Live TD control
rlcolab.widget("qlearning", map=["............", "............", "............", "SCCCCCCCCCCG"], algo="q", alpha=0.5, gamma=1.0, epsilon=0.1)

---
### 🧪 Exercise 3 · Q-learning on the cliff · ⭐ **150 XP**

Switch one symbol in the SARSA target and you get the most famous algorithm in RL.

1. `q_learning_update(Q, s, a, r, s2, alpha, gamma, terminated)` — in place, `Q[s,a] += α·(r + γ·maxₐ' Q[s2,a'] − Q[s,a])`; if `terminated` the target is just `r`. Return δ.
2. `q_learning(env, n_episodes, alpha=0.5, gamma=1.0, epsilon=0.1, seed=0)` → `(Q, returns)`. Same set-up as `sarsa` (`rng = np.random.default_rng(seed)`, zero table), but the loop is simpler: choose A ε-greedily from S, step, update, `s = s2`. No A' needed.

`epsilon_greedy`, a working `sarsa` and `greedy_rollout` are provided. The check requires the **optimal** greedy path (return −13) *and* that your agent's online return while exploring is worse than SARSA's — the paradox of S&B Example 6.6. **Run** plots both learning curves and both greedy paths.

> ✏️ **Edit** the next cell and run it (`Shift+Enter`) — that also runs its demo. Then run **✅ Check**. Stuck? The hints are right below the check.

In [ ]:
# ✏️ EXERCISE 06-td-learning/03-q-learning-cliff
import numpy as np
import rlhud
from rlenvs import CliffWalking


def q_learning_update(Q, s, a, r, s2, alpha, gamma, terminated):
    """One Q-learning update, in place:  Q(S,A) += alpha * [R + gamma * max_a' Q(S',a') - Q(S,A)].
    If terminated, the target is just R. Returns the TD error delta."""
    # TODO
    raise NotImplementedError


def q_learning(env, n_episodes, alpha=0.5, gamma=1.0, epsilon=0.1, seed=0):
    """Off-policy TD control: behave epsilon-greedily, learn about the greedy policy.
    Returns (Q, returns) where returns[i] is the total reward of episode i."""
    rng = np.random.default_rng(seed)
    Q = np.zeros((env.n_states, env.n_actions))
    returns = []
    for _ in range(n_episodes):
        # TODO: loop: choose A epsilon-greedily, step, q_learning_update, s <- s'
        #       stop on terminated or truncated; append the episode's total reward to `returns`
        raise NotImplementedError
    return Q, returns


# ---------------------------------------------------------------- provided (from the SARSA exercise)
def epsilon_greedy(Q, s, epsilon, rng):
    """Action for state s from the table Q (shape n_states x n_actions).
    With prob. epsilon a uniformly random action, else a greedy one (ties broken at random)."""
    if rng.random() < epsilon:
        return int(rng.integers(Q.shape[1]))
    best = np.flatnonzero(Q[s] == Q[s].max())
    return int(rng.choice(best))


def sarsa_update(Q, s, a, r, s2, a2, alpha, gamma, terminated):
    """One SARSA update, in place:  Q(S,A) += alpha * [R + gamma*Q(S',A') - Q(S,A)].
    If terminated, the target is just R. Returns the TD error delta."""
    target = r if terminated else r + gamma * Q[s2, a2]
    delta = target - Q[s, a]
    Q[s, a] += alpha * delta
    return delta


def sarsa(env, n_episodes, alpha=0.5, gamma=1.0, epsilon=0.1, seed=0):
    """On-policy TD control. Returns (Q, returns) where returns[i] is the total reward of episode i."""
    rng = np.random.default_rng(seed)
    Q = np.zeros((env.n_states, env.n_actions))
    returns = []
    for _ in range(n_episodes):
        s, _ = env.reset()
        a = epsilon_greedy(Q, s, epsilon, rng)
        G = 0.0
        while True:
            s2, r, terminated, truncated, _ = env.step(a)
            G += r
            a2 = epsilon_greedy(Q, s2, epsilon, rng)    # choose A' FIRST: it is used in the target...
            sarsa_update(Q, s, a, r, s2, a2, alpha, gamma, terminated)
            if terminated or truncated:
                break
            s, a = s2, a2                                # ...and it is the action we actually take next
        returns.append(G)
    return Q, returns


def greedy_rollout(env, Q, max_steps=100):
    """Follow argmax Q from the start (no exploration). Returns (total_reward, visited_states)."""
    s, _ = env.reset()
    total, path = 0.0, [s]
    for _ in range(max_steps):
        s, r, term, trunc, _ = env.step(int(np.argmax(Q[s])))
        total += r
        path.append(s)
        if term:
            break
    return total, path


def arrows(Q):
    """Greedy action per cell for rlhud.grid, with no arrow on the cliff and the goal."""
    g = np.argmax(Q, axis=1).reshape(4, 12).tolist()
    for c in range(1, 12):
        g[3][c] = None
    return g


if __name__ == "__main__":
    env = CliffWalking()
    q_Q, q_rets = q_learning(env, 500, alpha=0.5, epsilon=0.1, seed=0)
    s_Q, s_rets = sarsa(env, 500, alpha=0.5, epsilon=0.1, seed=0)
    clip = lambda xs: [max(x, -150) for x in xs]
    rlhud.plot(clip(q_rets), label="Q-learning", title="Online return while learning (ε = 0.1)", smooth=20)
    rlhud.plot(clip(s_rets), label="SARSA", title="Online return while learning (ε = 0.1)", smooth=20)

    for name, Q, rets in (("Q-learning", q_Q, q_rets), ("SARSA", s_Q, s_rets)):
        G, path = greedy_rollout(env, Q)
        visits = np.zeros(env.n_states)
        visits[path] = 1
        rlhud.grid(env.to_grid(visits), title=f"{name} greedy path: return {G:.0f}",
                   policy=arrows(Q), labels=env.labels(),
                   cmap="heat", fmt=".0f")
        rlhud.stat(f"{name}: greedy return", G)
        rlhud.stat(f"{name}: mean online return", round(float(np.mean(rets)), 1))
    print("Q-learning's greedy path is optimal (−13) but it falls off the cliff more while exploring.")

In [ ]:
#@title ✅ **Check** · Q-learning on the cliff (+150 XP) — tests the code you last ran above
rlcolab.check("06-td-learning/03-q-learning-cliff")

<details><summary>💡 <b>Hint 1</b> of 3 — try it before opening the next one</summary>

The behaviour policy (how you pick the action you take) is still ε-greedy. Only the *target* changes: it assumes you will act greedily from S' onwards.


<details><summary>💡 <b>Hint 2</b> of 3 — try it before opening the next one</summary>

`np.max(Q[s2])` is the value of the best action in S'. If `terminated`, there is no S' to bootstrap from: `target = r`.


<details><summary>💡 <b>Hint 3</b> of 3 — nearly gives it away</summary>

Loop body:

a = epsilon_greedy(Q, s, epsilon, rng)
s2, r, terminated, truncated, _ = env.step(a)
G += r
q_learning_update(Q, s, a, r, s2, alpha, gamma, terminated)
if terminated or truncated: break
s = s2

</details>

</details>

</details>

In [ ]:
#@title 🔒 Reference solution — double-click to reveal · running it replaces your version
rlcolab.solution_gate("06-td-learning/03-q-learning-cliff")  # obeys the Solutions setting in 00 · START HERE
# ✏️ EXERCISE 06-td-learning/03-q-learning-cliff
# Reference solution
import numpy as np
import rlhud
from rlenvs import CliffWalking


def q_learning_update(Q, s, a, r, s2, alpha, gamma, terminated):
    """One Q-learning update, in place:  Q(S,A) += alpha * [R + gamma * max_a' Q(S',a') - Q(S,A)].
    If terminated, the target is just R. Returns the TD error delta."""
    target = r if terminated else r + gamma * np.max(Q[s2])
    delta = target - Q[s, a]
    Q[s, a] += alpha * delta
    return delta


def q_learning(env, n_episodes, alpha=0.5, gamma=1.0, epsilon=0.1, seed=0):
    """Off-policy TD control: behave epsilon-greedily, learn about the greedy policy.
    Returns (Q, returns) where returns[i] is the total reward of episode i."""
    rng = np.random.default_rng(seed)
    Q = np.zeros((env.n_states, env.n_actions))
    returns = []
    for _ in range(n_episodes):
        s, _ = env.reset()
        G = 0.0
        while True:
            a = epsilon_greedy(Q, s, epsilon, rng)       # behaviour policy: epsilon-greedy
            s2, r, terminated, truncated, _ = env.step(a)
            G += r
            q_learning_update(Q, s, a, r, s2, alpha, gamma, terminated)   # target policy: greedy
            if terminated or truncated:
                break
            s = s2
        returns.append(G)
    return Q, returns


# ---------------------------------------------------------------- provided (from the SARSA exercise)
def epsilon_greedy(Q, s, epsilon, rng):
    """Action for state s from the table Q (shape n_states x n_actions).
    With prob. epsilon a uniformly random action, else a greedy one (ties broken at random)."""
    if rng.random() < epsilon:
        return int(rng.integers(Q.shape[1]))
    best = np.flatnonzero(Q[s] == Q[s].max())
    return int(rng.choice(best))


def sarsa_update(Q, s, a, r, s2, a2, alpha, gamma, terminated):
    """One SARSA update, in place:  Q(S,A) += alpha * [R + gamma*Q(S',A') - Q(S,A)].
    If terminated, the target is just R. Returns the TD error delta."""
    target = r if terminated else r + gamma * Q[s2, a2]
    delta = target - Q[s, a]
    Q[s, a] += alpha * delta
    return delta


def sarsa(env, n_episodes, alpha=0.5, gamma=1.0, epsilon=0.1, seed=0):
    """On-policy TD control. Returns (Q, returns) where returns[i] is the total reward of episode i."""
    rng = np.random.default_rng(seed)
    Q = np.zeros((env.n_states, env.n_actions))
    returns = []
    for _ in range(n_episodes):
        s, _ = env.reset()
        a = epsilon_greedy(Q, s, epsilon, rng)
        G = 0.0
        while True:
            s2, r, terminated, truncated, _ = env.step(a)
            G += r
            a2 = epsilon_greedy(Q, s2, epsilon, rng)    # choose A' FIRST: it is used in the target...
            sarsa_update(Q, s, a, r, s2, a2, alpha, gamma, terminated)
            if terminated or truncated:
                break
            s, a = s2, a2                                # ...and it is the action we actually take next
        returns.append(G)
    return Q, returns


def greedy_rollout(env, Q, max_steps=100):
    """Follow argmax Q from the start (no exploration). Returns (total_reward, visited_states)."""
    s, _ = env.reset()
    total, path = 0.0, [s]
    for _ in range(max_steps):
        s, r, term, trunc, _ = env.step(int(np.argmax(Q[s])))
        total += r
        path.append(s)
        if term:
            break
    return total, path


def arrows(Q):
    """Greedy action per cell for rlhud.grid, with no arrow on the cliff and the goal."""
    g = np.argmax(Q, axis=1).reshape(4, 12).tolist()
    for c in range(1, 12):
        g[3][c] = None
    return g


if __name__ == "__main__":
    env = CliffWalking()
    q_Q, q_rets = q_learning(env, 500, alpha=0.5, epsilon=0.1, seed=0)
    s_Q, s_rets = sarsa(env, 500, alpha=0.5, epsilon=0.1, seed=0)
    clip = lambda xs: [max(x, -150) for x in xs]
    rlhud.plot(clip(q_rets), label="Q-learning", title="Online return while learning (ε = 0.1)", smooth=20)
    rlhud.plot(clip(s_rets), label="SARSA", title="Online return while learning (ε = 0.1)", smooth=20)

    for name, Q, rets in (("Q-learning", q_Q, q_rets), ("SARSA", s_Q, s_rets)):
        G, path = greedy_rollout(env, Q)
        visits = np.zeros(env.n_states)
        visits[path] = 1
        rlhud.grid(env.to_grid(visits), title=f"{name} greedy path: return {G:.0f}",
                   policy=arrows(Q), labels=env.labels(),
                   cmap="heat", fmt=".0f")
        rlhud.stat(f"{name}: greedy return", G)
        rlhud.stat(f"{name}: mean online return", round(float(np.mean(rets)), 1))
    print("Q-learning's greedy path is optimal (−13) but it falls off the cliff more while exploring.")

> **💡 TIP** · **The cliff paradox**
>
> Q-learning's greedy path is optimal (−13) but its **online** return is worse than SARSA's,
> because it keeps walking the cliff edge while exploring and every so often slips off. Which
> algorithm is "better" depends on what you care about: the policy you end up with, or the
> reward you collect while learning. If ε decays to 0, both converge to the optimal path.

In [ ]:
#@title 🧠 **Knowledge check 2** · +25 XP — run, then pick an answer
rlcolab.quiz("06-td-learning#1", {"q": "A robot learns on real hardware where falls are expensive. Exploration will stay on during training. Which update better reflects the cost of its own exploratory slips?", "options": ["Q-learning, because it learns q*", "SARSA, because it evaluates the ε-greedy policy it actually follows", "Neither — TD cannot handle risk"], "answer": 1, "explain": "SARSA's target uses the action really taken, so the danger of random actions near the cliff is priced into Q. Q-learning assumes perfect greedy play from the next step on."})

## 05 · Expected SARSA

SARSA's target depends on the random draw of A'. Why sample when you know the policy? Expected
SARSA averages over it:

```text
Q(S, A) ← Q(S, A) + α [ R + γ Σₐ' π(a'|S') Q(S', a') − Q(S, A) ]
```

It moves in the same direction SARSA moves *on average*, with the variance from A' removed, so
it tolerates much larger step sizes (often α = 1 works on deterministic tasks like the cliff).
If π is greedy, the sum is just the max: **Q-learning is the special case of Expected SARSA with
a greedy target policy.** With an ε-greedy π it is on-policy; with any other π it is a
general off-policy method.

In [ ]:
#@title 🧠 **Knowledge check 3** · +25 XP — run, then pick an answer
rlcolab.quiz("06-td-learning#2", {"q": "Q(S', ·) = [2, 0, −1, 1], π is ε-greedy with ε = 0.2 over 4 actions. What is Σₐ' π(a'|S') Q(S', a')?", "options": ["2.0", "1.7", "0.5", "1.55"], "answer": 1, "explain": "Greedy action gets 1 − 0.2 + 0.05 = 0.85, the others 0.05: 0.85·2 + 0.05·(0 − 1 + 1) = 1.7."})

## 06 · Maximisation bias and Double Q-learning

The max in Q-learning hides a trap. Suppose every action in a state has true value 0 but our
estimates are noisy: some come out at +0.3, some at −0.4. The max of the *estimates* is
positive even though the max of the *true values* is 0. Using max of noisy estimates as an
estimate of the max is **biased upwards** — maximisation bias.

The fix is to decouple **selecting** the best action from **evaluating** it. Keep two
independent tables Q₁ and Q₂. On each step flip a coin; if heads:

```text
A* = argmaxₐ Q₁(S', a)        Q₁(S, A) ← Q₁(S, A) + α [ R + γ Q₂(S', A*) − Q₁(S, A) ]
```

and the mirror image if tails. Q₂'s noise is independent of the noise that made A* look good
in Q₁, so the estimate is unbiased. Behave ε-greedily with respect to Q₁ + Q₂. The same idea,
"one network selects, the other evaluates", becomes **Double DQN** in the deep RL chapter.

---
### 🧪 Exercise 4 · Double Q-learning vs maximisation bias · ⭐ **150 XP**

In the MDP of Sutton & Barto Example 6.7, going LEFT from A leads to B, where every action pays Normal(−0.1, 1). LEFT is a bad idea on average — but the **max** of ten noisy estimates in B is almost always positive, so Q-learning is lured left.

1. `double_q_update(Q1, Q2, s, a, r, s2, alpha, gamma, terminated, update_first)` — `Q1`, `Q2` are dicts `{state: array}`. If `update_first`: **select** `a* = argmax Q1[s2]`, **evaluate** it with `Q2[s2][a*]`, and update `Q1[s][a]` towards `r + γ·Q2[s2][a*]`. Otherwise do the mirror image, updating Q2. Terminal ⇒ target `r`. Return δ.
2. `double_q_learning(env, n_episodes, alpha=0.1, gamma=1.0, epsilon=0.1, seed=0)` — create `rng = np.random.default_rng(seed)` and zero tables `{s: np.zeros(env.n_actions(s)) for s in range(env.n_states)}`. Act with `epsilon_greedy(Q1[s] + Q2[s], epsilon, rng)`, and each step flip a coin `rng.random() < 0.5` to choose which table to update. Return a list with one boolean per episode: was the action taken in `env.A` equal to `env.LEFT`?

The env, `epsilon_greedy` and a plain `q_learning_bias` with the same output format are provided. **Run** reproduces Figure 6.5.

> ✏️ **Edit** the next cell and run it (`Shift+Enter`) — that also runs its demo. Then run **✅ Check**. Stuck? The hints are right below the check.

In [ ]:
# ✏️ EXERCISE 06-td-learning/04-double-q
import numpy as np
import rlhud


def double_q_update(Q1, Q2, s, a, r, s2, alpha, gamma, terminated, update_first):
    """One Double Q-learning update, in place.

    Q1, Q2 : dicts {state: np.array of action values}
    update_first=True : pick a* = argmax Q1[s2], evaluate it with Q2, update Q1[s][a]
    update_first=False: the mirror image (argmax with Q2, evaluate with Q1, update Q2[s][a])
    If terminated, the target is just r. Returns the TD error delta.
    """
    # TODO
    raise NotImplementedError


def double_q_learning(env, n_episodes, alpha=0.1, gamma=1.0, epsilon=0.1, seed=0):
    """Double Q-learning on `env`. Behaviour: epsilon-greedy on Q1 + Q2.
    Returns a list of booleans: did the agent go LEFT from state A in episode i?"""
    rng = np.random.default_rng(seed)
    Q1 = {s: np.zeros(env.n_actions(s)) for s in range(env.n_states)}
    Q2 = {s: np.zeros(env.n_actions(s)) for s in range(env.n_states)}
    went_left = []
    for _ in range(n_episodes):
        # TODO: act epsilon-greedily on Q1[s] + Q2[s]; record whether you went LEFT from env.A;
        #       step; double_q_update with a fair coin flip; stop on terminated or truncated
        raise NotImplementedError
    return went_left


# ---------------------------------------------------------------- provided
class MaximizationBiasMDP:
    """Sutton & Barto Example 6.7.

        terminal <--LEFT-- [B] <--LEFT-- [A] --RIGHT--> terminal (r = 0)
                  (r = 0)

    From A: RIGHT ends the episode with 0; LEFT moves to B with 0.
    From B: any of `n_b` actions ends the episode with reward ~ Normal(-0.1, 1).
    So LEFT is worse on average (−0.1 < 0) — but the max over noisy estimates in B looks positive.
    """
    A, B, T = 0, 1, 2
    LEFT, RIGHT = 0, 1
    n_states = 3

    def __init__(self, n_b=10, seed=None):
        self.n_b = n_b
        self.rng = np.random.default_rng(seed)

    def n_actions(self, s):
        return 2 if s == self.A else (self.n_b if s == self.B else 1)

    def reset(self, seed=None):
        if seed is not None:
            self.rng = np.random.default_rng(seed)
        self.s = self.A
        return self.s, {}

    def step(self, a):
        if self.s == self.A:
            if a == self.LEFT:
                self.s = self.B
                return self.B, 0.0, False, False, {}
            self.s = self.T
            return self.T, 0.0, True, False, {}
        self.s = self.T
        return self.T, float(self.rng.normal(-0.1, 1.0)), True, False, {}


def epsilon_greedy(q, epsilon, rng):
    """q: array of action values for one state. Random ties."""
    if rng.random() < epsilon:
        return int(rng.integers(len(q)))
    return int(rng.choice(np.flatnonzero(q == q.max())))


def q_learning_bias(env, n_episodes, alpha=0.1, gamma=1.0, epsilon=0.1, seed=0):
    """Plain Q-learning on the same MDP, for comparison. Same return format."""
    rng = np.random.default_rng(seed)
    Q = {s: np.zeros(env.n_actions(s)) for s in range(env.n_states)}
    went_left = []
    for _ in range(n_episodes):
        s, _ = env.reset()
        while True:
            a = epsilon_greedy(Q[s], epsilon, rng)
            if s == env.A:
                went_left.append(a == env.LEFT)
            s2, r, term, trunc, _ = env.step(a)
            target = r if term else r + gamma * Q[s2].max()
            Q[s][a] += alpha * (target - Q[s][a])
            if term or trunc:
                break
            s = s2
    return went_left


if __name__ == "__main__":
    RUNS, EPISODES = 500, 300
    curves = {"Q-learning": np.zeros(EPISODES), "Double Q-learning": np.zeros(EPISODES)}
    for run in range(RUNS):
        env = MaximizationBiasMDP(seed=run)
        curves["Q-learning"] += np.array(q_learning_bias(env, EPISODES, seed=run)) / RUNS
        curves["Double Q-learning"] += np.array(double_q_learning(env, EPISODES, seed=run)) / RUNS
        rlhud.progress(run, RUNS, "Runs")
    for name, c in curves.items():
        rlhud.plot(100 * c, label=name, title="% LEFT actions from A (avg of 500 runs)",
                   xlabel="episode", ylabel="% left")
        rlhud.stat(f"{name}: % left, episodes 1–50", round(100 * float(c[:50].mean()), 1))
    rlhud.plot([5.0] * EPISODES, label="optimal (ε/2 = 5 %)", title="% LEFT actions from A (avg of 500 runs)")
    print("Q-learning is fooled by max over noisy estimates; Double Q-learning is not.")

In [ ]:
#@title ✅ **Check** · Double Q-learning vs maximisation bias (+150 XP) — tests the code you last ran above
rlcolab.check("06-td-learning/04-double-q")

<details><summary>💡 <b>Hint 1</b> of 3 — try it before opening the next one</summary>

The bias comes from using the *same* noisy numbers to both choose the best action and estimate its value. Two independent tables break that link: one chooses, the other judges.


<details><summary>💡 <b>Hint 2</b> of 3 — try it before opening the next one</summary>

A clean trick: `A, B = (Q1, Q2) if update_first else (Q2, Q1)`, then write the update once in terms of A (updated, selects) and B (evaluates).


<details><summary>💡 <b>Hint 3</b> of 3 — nearly gives it away</summary>

Episode loop:

s, _ = env.reset()
while True:
    a = epsilon_greedy(Q1[s] + Q2[s], epsilon, rng)
    if s == env.A: went_left.append(a == env.LEFT)
    s2, r, terminated, truncated, _ = env.step(a)
    double_q_update(Q1, Q2, s, a, r, s2, alpha, gamma, terminated, rng.random() < 0.5)
    if terminated or truncated: break
    s = s2

</details>

</details>

</details>

In [ ]:
#@title 🔒 Reference solution — double-click to reveal · running it replaces your version
rlcolab.solution_gate("06-td-learning/04-double-q")  # obeys the Solutions setting in 00 · START HERE
# ✏️ EXERCISE 06-td-learning/04-double-q
# Reference solution
import numpy as np
import rlhud


def double_q_update(Q1, Q2, s, a, r, s2, alpha, gamma, terminated, update_first):
    """One Double Q-learning update, in place.

    Q1, Q2 : dicts {state: np.array of action values}
    update_first=True : pick a* = argmax Q1[s2], evaluate it with Q2, update Q1[s][a]
    update_first=False: the mirror image (argmax with Q2, evaluate with Q1, update Q2[s][a])
    If terminated, the target is just r. Returns the TD error delta.
    """
    A, B = (Q1, Q2) if update_first else (Q2, Q1)
    if terminated:
        target = r
    else:
        a_star = int(np.argmax(A[s2]))           # SELECT with the table being updated...
        target = r + gamma * B[s2][a_star]       # ...EVALUATE with the other one
    delta = target - A[s][a]
    A[s][a] += alpha * delta
    return delta


def double_q_learning(env, n_episodes, alpha=0.1, gamma=1.0, epsilon=0.1, seed=0):
    """Double Q-learning on `env`. Behaviour: epsilon-greedy on Q1 + Q2.
    Returns a list of booleans: did the agent go LEFT from state A in episode i?"""
    rng = np.random.default_rng(seed)
    Q1 = {s: np.zeros(env.n_actions(s)) for s in range(env.n_states)}
    Q2 = {s: np.zeros(env.n_actions(s)) for s in range(env.n_states)}
    went_left = []
    for _ in range(n_episodes):
        s, _ = env.reset()
        while True:
            a = epsilon_greedy(Q1[s] + Q2[s], epsilon, rng)
            if s == env.A:
                went_left.append(a == env.LEFT)
            s2, r, terminated, truncated, _ = env.step(a)
            double_q_update(Q1, Q2, s, a, r, s2, alpha, gamma, terminated, rng.random() < 0.5)
            if terminated or truncated:
                break
            s = s2
    return went_left


# ---------------------------------------------------------------- provided
class MaximizationBiasMDP:
    """Sutton & Barto Example 6.7.

        terminal <--LEFT-- [B] <--LEFT-- [A] --RIGHT--> terminal (r = 0)
                  (r = 0)

    From A: RIGHT ends the episode with 0; LEFT moves to B with 0.
    From B: any of `n_b` actions ends the episode with reward ~ Normal(-0.1, 1).
    So LEFT is worse on average (−0.1 < 0) — but the max over noisy estimates in B looks positive.
    """
    A, B, T = 0, 1, 2
    LEFT, RIGHT = 0, 1
    n_states = 3

    def __init__(self, n_b=10, seed=None):
        self.n_b = n_b
        self.rng = np.random.default_rng(seed)

    def n_actions(self, s):
        return 2 if s == self.A else (self.n_b if s == self.B else 1)

    def reset(self, seed=None):
        if seed is not None:
            self.rng = np.random.default_rng(seed)
        self.s = self.A
        return self.s, {}

    def step(self, a):
        if self.s == self.A:
            if a == self.LEFT:
                self.s = self.B
                return self.B, 0.0, False, False, {}
            self.s = self.T
            return self.T, 0.0, True, False, {}
        self.s = self.T
        return self.T, float(self.rng.normal(-0.1, 1.0)), True, False, {}


def epsilon_greedy(q, epsilon, rng):
    """q: array of action values for one state. Random ties."""
    if rng.random() < epsilon:
        return int(rng.integers(len(q)))
    return int(rng.choice(np.flatnonzero(q == q.max())))


def q_learning_bias(env, n_episodes, alpha=0.1, gamma=1.0, epsilon=0.1, seed=0):
    """Plain Q-learning on the same MDP, for comparison. Same return format."""
    rng = np.random.default_rng(seed)
    Q = {s: np.zeros(env.n_actions(s)) for s in range(env.n_states)}
    went_left = []
    for _ in range(n_episodes):
        s, _ = env.reset()
        while True:
            a = epsilon_greedy(Q[s], epsilon, rng)
            if s == env.A:
                went_left.append(a == env.LEFT)
            s2, r, term, trunc, _ = env.step(a)
            target = r if term else r + gamma * Q[s2].max()
            Q[s][a] += alpha * (target - Q[s][a])
            if term or trunc:
                break
            s = s2
    return went_left


if __name__ == "__main__":
    RUNS, EPISODES = 500, 300
    curves = {"Q-learning": np.zeros(EPISODES), "Double Q-learning": np.zeros(EPISODES)}
    for run in range(RUNS):
        env = MaximizationBiasMDP(seed=run)
        curves["Q-learning"] += np.array(q_learning_bias(env, EPISODES, seed=run)) / RUNS
        curves["Double Q-learning"] += np.array(double_q_learning(env, EPISODES, seed=run)) / RUNS
        rlhud.progress(run, RUNS, "Runs")
    for name, c in curves.items():
        rlhud.plot(100 * c, label=name, title="% LEFT actions from A (avg of 500 runs)",
                   xlabel="episode", ylabel="% left")
        rlhud.stat(f"{name}: % left, episodes 1–50", round(100 * float(c[:50].mean()), 1))
    rlhud.plot([5.0] * EPISODES, label="optimal (ε/2 = 5 %)", title="% LEFT actions from A (avg of 500 runs)")
    print("Q-learning is fooled by max over noisy estimates; Double Q-learning is not.")

## 07 · Pitfalls

> **⚠️ WARNING** · **Common TD bugs**
>
> - **Bootstrapping from a terminal state.** If `terminated`, the target is just R. Using
>   Q(S', ·) of a terminal state is wrong unless you guarantee it stays 0.
> - **Truncated is not terminated.** A time limit cuts the episode, but the state still has
>   a future. Keep bootstrapping on `truncated`; only drop the bootstrap on `terminated`.
> - **SARSA's A' mismatch.** Choose A' once, use it in the target, and then *take* it. Re-drawing
>   a fresh action at the top of the loop silently turns SARSA into something else.
> - **argmax ties.** With a zero-initialised table, `np.argmax` always picks action 0. Break ties
>   at random or your agent may never leave the first wall it walks into.
> - **Step size too large for noise.** α = 0.5 is fine on the deterministic cliff, terrible on a
>   noisy task. If curves oscillate, lower α.

## 08 · Summary

- **TD(0)** updates V(S) towards R + γV(S') after every step; the gap is the **TD error δ**.
- TD is model-free like MC, bootstraps like DP, works online and on continuing tasks, and is
  usually faster than MC thanks to lower variance.
- **SARSA** (on-policy) bootstraps from the action actually taken; **Q-learning** (off-policy)
  from the greedy one; **Expected SARSA** from the policy's average.
- On the cliff, SARSA learns the safe path, Q-learning the optimal-but-risky one.
- **Double Q-learning** removes maximisation bias by letting one table select and the other evaluate.

Next: why look only *one* step ahead? n-step methods and eligibility traces fill the gap between
TD and MC, and Dyna shows how a learned model can multiply every real step into many imagined ones.

---
## 🏁 Debrief
Run the cell below for your mission status, then open the next notebook.

In [ ]:
#@title 🏁 **Mission status** — run to refresh
rlcolab.lesson_footer("06-td-learning")